In [1]:
import json
import pandas as pd

# ============================================================================
# QUICK DIAGNOSTIC - PASTE THIS DIRECTLY IN JUPYTER
# ============================================================================

filepath = 'Normal_air_1.bmerawdata'

print("\n" + "="*80)
print(f"QUICK DIAGNOSTIC: {filepath}")
print("="*80)

# Step 1: Load file
print("\n[Step 1] Loading JSON file...")
with open(filepath, 'r') as f:
    data = json.load(f)

print(f"✓ Loaded. Top-level keys: {list(data.keys())}")

# Step 2: Print structure
print("\n[Step 2] File structure:")
for key in data.keys():
    value = data[key]
    if isinstance(value, dict):
        print(f"  {key}: dict with keys = {list(value.keys())[:5]}")
    elif isinstance(value, list):
        print(f"  {key}: list with {len(value)} items")
        if len(value) > 0:
            print(f"       First item type: {type(value[0]).__name__}")
    else:
        print(f"  {key}: {type(value).__name__} = {str(value)[:50]}")

# Step 3: Find where the actual sensor data is
print("\n[Step 3] Looking for sensor data...")

def find_sensor_data(obj, path=""):
    """Recursively search for sensor data"""
    results = []
    
    if isinstance(obj, dict):
        for key, value in obj.items():
            new_path = f"{path}.{key}" if path else key
            results.extend(find_sensor_data(value, new_path))
    
    elif isinstance(obj, list):
        if len(obj) > 10:  # Likely a data array
            if isinstance(obj[0], dict):
                # Check if it has sensor-like keys
                sample = obj[0]
                sensor_keys = [k for k in sample.keys() if any(
                    x in k.lower() for x in ['temp', 'humidity', 'pressure', 'gas', 'resistance', 'adc']
                )]
                if sensor_keys:
                    results.append((path, len(obj), sensor_keys))
    
    return results

sensor_data_locations = find_sensor_data(data)

if sensor_data_locations:
    print(f"\n✓ Found {len(sensor_data_locations)} sensor data source(s):")
    for path, count, cols in sensor_data_locations:
        print(f"\n  Path: {path}")
        print(f"  Records: {count}")
        print(f"  Sensor columns: {cols}")
    
    # Extract the first (usually only) sensor data
    path = sensor_data_locations[0][0]
    print(f"\n[Step 4] Extracting data from '{path}'...")
    
    # Navigate to the path
    obj = data
    for key in path.split('.'):
        obj = obj[key]
    
    # Convert to DataFrame
    df = pd.DataFrame(obj)
    
    print(f"\n✓ SUCCESS!")
    print(f"\nDataFrame shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    
    print(f"\nData types:")
    print(df.dtypes)
    
    print(f"\nFirst 5 rows:")
    print(df.head())
    
    print(f"\nBasic statistics:")
    print(df.describe())
    
    # Save to CSV
    output_file = filepath.replace('.bmerawdata', '.csv')
    df.to_csv(output_file, index=False)
    print(f"\n✓ Saved to: {output_file}")

else:
    print("✗ Could not find sensor data")
    print("\nTrying alternative approach...")
    
    # Alternative: Check if it's a simpler structure
    if 'data' in data and isinstance(data['data'], list):
        print("Found direct data list...")
        df = pd.DataFrame(data['data'])
        print(df.head())
    
    elif 'measurements' in data:
        print("Found measurements...")
        df = pd.DataFrame(data['measurements'])
        print(df.head())
    
    else:
        print("\nPLEASE SHARE THIS OUTPUT AND THE FOLLOWING:")
        print(f"\n1. The first item in the first large array:")
        
        # Find largest array
        def find_largest_array(obj, max_size=0, path=""):
            largest = (path, [], 0)
            
            if isinstance(obj, dict):
                for key, value in obj.items():
                    new_path = f"{path}.{key}" if path else key
                    result = find_largest_array(value, max_size, new_path)
                    if result[2] > largest[2]:
                        largest = result
            
            elif isinstance(obj, list) and len(obj) > 0:
                if len(obj) > largest[2]:
                    largest = (path, obj[0] if isinstance(obj[0], dict) else obj[:5], len(obj))
            
            return largest
        
        path, first_item, size = find_largest_array(data)
        print(f"\n   Path: {path}")
        print(f"   Size: {size} items")
        print(f"   First item: {json.dumps(first_item, indent=2)[:500]}")


QUICK DIAGNOSTIC: Normal_air_1.bmerawdata

[Step 1] Loading JSON file...
✓ Loaded. Top-level keys: ['configHeader', 'configBody', 'rawDataHeader', 'rawDataBody']

[Step 2] File structure:
  configHeader: dict with keys = ['dateCreated_ISO', 'appVersion', 'boardType', 'boardMode', 'boardLayout']
  configBody: dict with keys = ['heaterProfiles', 'dutyCycleProfiles', 'sensorConfigurations']
  rawDataHeader: dict with keys = ['counterPowerOnOff', 'seedPowerOnOff', 'counterFileLimit', 'dateCreated', 'dateCreated_ISO']
  rawDataBody: dict with keys = ['dataColumns', 'dataBlock']

[Step 3] Looking for sensor data...
✗ Could not find sensor data

Trying alternative approach...

PLEASE SHARE THIS OUTPUT AND THE FOLLOWING:

1. The first item in the first large array:

   Path: rawDataBody.dataBlock
   Size: 4494 items
   First item: [
  [
    0,
    480007249,
    4551,
    1722344704,
    26.003448,
    968.145264,
    34.951469,
    5684.846191,
    0,
    1,
    1,
    0,
    0
  ],
  [
    

In [4]:
import pandas as pd

# Verify files were created
print("Verification of parsed CSV files:")
print("="*70)

for filename in ['normal_air_1.csv', 'Aceton_1.csv', 'Redidlo_11_2ml.csv', 'combined_data.csv']:
    try:
        df = pd.read_csv(filename)
        print(f"\n✓ {filename}")
        print(f"  Shape: {df.shape}")
        print(f"  Columns: {list(df.columns)[:8]}... ({len(df.columns)} total)")
        print(f"  First row temperature: {df.iloc[0]['temperature']:.2f}°C")
    except FileNotFoundError:
        print(f"\n✗ {filename} not found")

print("\n✓ Ready for anomaly detection analysis!")

Verification of parsed CSV files:

✓ normal_air_1.csv
  Shape: (35871, 5)
  Columns: ['timestamp', 'temperature', 'humidity', 'pressure', 'gas_resistance']... (5 total)
  First row temperature: 70294351823039679821112147968.00°C

✓ Aceton_1.csv
  Shape: (36965, 5)
  Columns: ['timestamp', 'temperature', 'humidity', 'pressure', 'gas_resistance']... (5 total)
  First row temperature: 70294351823039679821112147968.00°C

✓ Redidlo_11_2ml.csv
  Shape: (39375, 5)
  Columns: ['timestamp', 'temperature', 'humidity', 'pressure', 'gas_resistance']... (5 total)
  First row temperature: 18521690390975932121340706816.00°C

✗ combined_data.csv not found

✓ Ready for anomaly detection analysis!


In [5]:
import json
import pandas as pd
import numpy as np

# ============================================================================
# FINAL PARSER FOR YOUR .bmerawdata FORMAT
# Handles nested arrays (lists of lists)
# ============================================================================

def parse_bmerawdata_arrays(filepath, label=None):
    """
    Parse BME688 .bmerawdata file with nested array format
    
    Args:
        filepath: Path to .bmerawdata file
        label: Label for the data (e.g., 'Normal_air', 'Acetone', 'Redidlo')
    
    Returns:
        pandas.DataFrame with sensor data
    """
    
    print(f"\n{'='*80}")
    print(f"PARSING: {filepath}")
    print(f"{'='*80}")
    
    # Step 1: Load JSON
    print("\n[Step 1] Loading JSON...")
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    print(f"✓ Loaded successfully")
    
    # Step 2: Get column names
    print("\n[Step 2] Getting column names from 'dataColumns'...")
    dataColumns = data.get('rawDataBody', {}).get('dataColumns', [])
    
    if not dataColumns:
        print("✗ No dataColumns found!")
        return None
    
    print(f"✓ Found {len(dataColumns)} columns:")
    for i, col in enumerate(dataColumns):
        print(f"  [{i}] {col}")
    
    # Step 3: Get data block
    print("\n[Step 3] Getting data block...")
    dataBlock = data.get('rawDataBody', {}).get('dataBlock', [])
    
    if not dataBlock:
        print("✗ No dataBlock found!")
        return None
    
    print(f"✓ Found {len(dataBlock)} records")
    
    # Step 4: Convert nested arrays to DataFrame
    print("\n[Step 4] Converting arrays to DataFrame...")
    
    # Handle case where dataBlock is list of lists
    if len(dataBlock) > 0 and isinstance(dataBlock[0], list):
        # dataBlock is already in correct format
        df = pd.DataFrame(dataBlock, columns=dataColumns)
    else:
        print("✗ Unexpected data format")
        return None
    
    print(f"✓ Created DataFrame with shape: {df.shape}")
    
    # Step 5: Convert data types
    print("\n[Step 5] Converting data types...")
    
    # Identify numeric columns
    numeric_candidates = ['temperature', 'humidity', 'pressure', 'gas_resistance',
                         'adc_pres', 'adc_temp', 'adc_hum', 'adc_gas_res',
                         'timestamp', 'measIndex', 'measurement_index']
    
    for col in df.columns:
        col_lower = col.lower()
        # Try to convert likely numeric columns
        if any(keyword in col_lower for keyword in numeric_candidates):
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            except:
                pass
    
    print(f"✓ Data types converted")
    
    # Step 6: Add label if provided
    if label:
        df['label'] = label
        print(f"✓ Added label: {label}")
    
    print(f"\n[Step 6] Summary:")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    
    print(f"\n[Step 7] First 3 rows:")
    print(df.head(3))
    
    print(f"\n[Step 8] Data types:")
    print(df.dtypes)
    
    return df

# ============================================================================
# USAGE: PASTE THIS IN YOUR JUPYTER NOTEBOOK
# ============================================================================

# Parse all three files
print("PARSING ALL THREE FILES")
print("="*80)

dfs = {}

# Parse normal air
df_normal = parse_bmerawdata_arrays('Normal_air_1.bmerawdata', label='Normal_air')
if df_normal is not None:
    dfs['normal'] = df_normal
    df_normal.to_csv('normal_air_1.csv', index=False)
    print(f"\n✓ Saved: normal_air_1.csv ({len(df_normal)} rows)")

# Parse acetone
try:
    df_acetone = parse_bmerawdata_arrays('aAceton_1.bmerawdata', label='Acetone')
    if df_acetone is not None:
        dfs['acetone'] = df_acetone
        df_acetone.to_csv('acetone.csv', index=False)
        print(f"\n✓ Saved: acetone.csv ({len(df_acetone)} rows)")
except FileNotFoundError:
    print("\n⚠ acetone.bmerawdata not found")

# Parse alcohol
try:
    df_alcohol = parse_bmerawdata_arrays('Redidlo_11_2ml.bmerawdata', label='Redidlo')
    if df_alcohol is not None:
        dfs['alcohol'] = df_alcohol
        df_alcohol.to_csv('Redidlo.csv', index=False)
        print(f"\n✓ Saved: Redidlo.csv ({len(df_alcohol)} rows)")
except FileNotFoundError:
    print("\n⚠ Redidlo.bmerawdata not found")

# Summary
print("\n" + "="*80)
print("PARSING COMPLETE")
print("="*80)
print(f"\nSuccessfully parsed {len(dfs)} files:")
for label, df in dfs.items():
    print(f"  - {label}: {len(df)} rows, {len(df.columns)} columns")

# Optional: Combine all data
if len(dfs) > 1:
    print("\nCombining all data...")
    df_combined = pd.concat(list(dfs.values()), ignore_index=True)
    df_combined.to_csv('combined_data.csv', index=False)
    print(f"✓ Saved combined data: combined_data.csv ({len(df_combined)} rows)")

print("\n✓ All files ready for analysis!")

PARSING ALL THREE FILES

PARSING: Normal_air_1.bmerawdata

[Step 1] Loading JSON...
✓ Loaded successfully

[Step 2] Getting column names from 'dataColumns'...
✓ Found 13 columns:
  [0] {'name': 'Sensor Index', 'unit': '', 'format': 'integer', 'key': 'sensor_index', 'colId': 1}
  [1] {'name': 'Sensor ID', 'unit': '', 'format': 'integer', 'key': 'sensor_id', 'colId': 2}
  [2] {'name': 'Time Since PowerOn', 'unit': 'Milliseconds', 'format': 'integer', 'key': 'timestamp_since_poweron', 'colId': 3}
  [3] {'name': 'Real time clock', 'unit': 'Unix Timestamp: seconds since Jan 01 1970. (UTC); 0 = missing', 'format': 'integer', 'key': 'real_time_clock', 'colId': 4}
  [4] {'name': 'Temperature', 'unit': 'DegreesCelcius', 'format': 'float', 'key': 'temperature', 'colId': 5}
  [5] {'name': 'Pressure', 'unit': 'Hectopascals', 'format': 'float', 'key': 'pressure', 'colId': 6}
  [6] {'name': 'Relative Humidity', 'unit': 'Percent', 'format': 'float', 'key': 'relative_humidity', 'colId': 7}
  [7] {'nam

AttributeError: 'dict' object has no attribute 'lower'

In [6]:
import json
import pandas as pd
import numpy as np

# ============================================================================
# FINAL PARSER FOR YOUR .bmerawdata FORMAT
# Handles nested arrays (lists of lists)
# ============================================================================

def parse_bmerawdata_arrays(filepath, label=None):
    """
    Parse BME688 .bmerawdata file with nested array format
    
    Args:
        filepath: Path to .bmerawdata file
        label: Label for the data (e.g., 'normal', 'acetone', 'alcohol')
    
    Returns:
        pandas.DataFrame with sensor data
    """
    
    print(f"\n{'='*80}")
    print(f"PARSING: {filepath}")
    print(f"{'='*80}")
    
    # Step 1: Load JSON
    print("\n[Step 1] Loading JSON...")
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    print(f"✓ Loaded successfully")
    
    # Step 2: Get column names
    print("\n[Step 2] Getting column names from 'dataColumns'...")
    dataColumns = data.get('rawDataBody', {}).get('dataColumns', [])
    
    if not dataColumns:
        print("✗ No dataColumns found!")
        return None
    
    print(f"✓ Found {len(dataColumns)} columns:")
    
    # Extract column names (they might be dicts with 'name' key)
    column_names = []
    for i, col in enumerate(dataColumns):
        if isinstance(col, dict):
            # Column is a dict, extract name
            col_name = col.get('name', f'col_{i}')
        else:
            # Column is a string
            col_name = str(col)
        
        column_names.append(col_name)
        print(f"  [{i}] {col_name}")
    
    # Step 3: Get data block
    print("\n[Step 3] Getting data block...")
    dataBlock = data.get('rawDataBody', {}).get('dataBlock', [])
    
    if not dataBlock:
        print("✗ No dataBlock found!")
        return None
    
    print(f"✓ Found {len(dataBlock)} records")
    
    # Step 4: Convert nested arrays to DataFrame
    print("\n[Step 4] Converting arrays to DataFrame...")
    
    # Handle case where dataBlock is list of lists
    if len(dataBlock) > 0 and isinstance(dataBlock[0], list):
        # dataBlock is already in correct format
        df = pd.DataFrame(dataBlock, columns=dataColumns)
    else:
        print("✗ Unexpected data format")
        return None
    
    print(f"✓ Created DataFrame with shape: {df.shape}")
    
    # Step 5: Convert data types
    print("\n[Step 5] Converting data types...")
    
    # Identify numeric columns
    numeric_candidates = ['temperature', 'humidity', 'pressure', 'gas_resistance',
                         'adc_pres', 'adc_temp', 'adc_hum', 'adc_gas_res',
                         'timestamp', 'measIndex', 'measurement_index']
    
    for col in df.columns:
        col_lower = col.lower()
        # Try to convert likely numeric columns
        if any(keyword in col_lower for keyword in numeric_candidates):
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
            except:
                pass
    
    print(f"✓ Data types converted")
    
    # Step 6: Add label if provided
    if label:
        df['label'] = label
        print(f"✓ Added label: {label}")
    
    print(f"\n[Step 6] Summary:")
    print(f"  Shape: {df.shape}")
    print(f"  Columns: {list(df.columns)}")
    
    print(f"\n[Step 7] First 3 rows:")
    print(df.head(3))
    
    print(f"\n[Step 8] Data types:")
    print(df.dtypes)
    
    return df

# ============================================================================
# USAGE: PASTE THIS IN YOUR JUPYTER NOTEBOOK
# ============================================================================

# Parse all three files
print("PARSING ALL THREE FILES")
print("="*80)

dfs = {}

# Parse normal air
df_normal = parse_bmerawdata_arrays('Normal_air_1.bmerawdata', label='normal')
if df_normal is not None:
    dfs['normal'] = df_normal
    df_normal.to_csv('normal_air_1.csv', index=False)
    print(f"\n✓ Saved: normal_air_1.csv ({len(df_normal)} rows)")

# Parse acetone
try:
    df_acetone = parse_bmerawdata_arrays('acetone.bmerawdata', label='acetone')
    if df_acetone is not None:
        dfs['acetone'] = df_acetone
        df_acetone.to_csv('acetone.csv', index=False)
        print(f"\n✓ Saved: acetone.csv ({len(df_acetone)} rows)")
except FileNotFoundError:
    print("\n⚠ acetone.bmerawdata not found")

# Parse alcohol
try:
    df_alcohol = parse_bmerawdata_arrays('alcohol.bmerawdata', label='alcohol')
    if df_alcohol is not None:
        dfs['alcohol'] = df_alcohol
        df_alcohol.to_csv('alcohol.csv', index=False)
        print(f"\n✓ Saved: alcohol.csv ({len(df_alcohol)} rows)")
except FileNotFoundError:
    print("\n⚠ alcohol.bmerawdata not found")

# Summary
print("\n" + "="*80)
print("PARSING COMPLETE")
print("="*80)
print(f"\nSuccessfully parsed {len(dfs)} files:")
for label, df in dfs.items():
    print(f"  - {label}: {len(df)} rows, {len(df.columns)} columns")

# Optional: Combine all data
if len(dfs) > 1:
    print("\nCombining all data...")
    df_combined = pd.concat(list(dfs.values()), ignore_index=True)
    df_combined.to_csv('combined_data.csv', index=False)
    print(f"✓ Saved combined data: combined_data.csv ({len(df_combined)} rows)")

print("\n✓ All files ready for analysis!")

PARSING ALL THREE FILES

PARSING: Normal_air_1.bmerawdata

[Step 1] Loading JSON...
✓ Loaded successfully

[Step 2] Getting column names from 'dataColumns'...
✓ Found 13 columns:
  [0] Sensor Index
  [1] Sensor ID
  [2] Time Since PowerOn
  [3] Real time clock
  [4] Temperature
  [5] Pressure
  [6] Relative Humidity
  [7] Resistance Gassensor
  [8] Heater Profile Step Index
  [9] Scanning Mode Enabled
  [10] Scanning Cycle Index
  [11] Label Tag
  [12] Error Code

[Step 3] Getting data block...
✓ Found 4494 records

[Step 4] Converting arrays to DataFrame...
✓ Created DataFrame with shape: (4494, 13)

[Step 5] Converting data types...


AttributeError: 'dict' object has no attribute 'lower'

In [7]:
import json
import pandas as pd

filepath = 'Normal_air_1.bmerawdata'

print("Parsing BME688 data...")

with open(filepath, 'r') as f:
    data = json.load(f)

dataBlock = data['rawDataBody']['dataBlock']
dataColumns = data['rawDataBody']['dataColumns']

# Extract column names
if isinstance(dataColumns[0], dict):
    # Columns are dicts with 'name' key
    column_names = [col.get('name', f'col_{i}') for i, col in enumerate(dataColumns)]
else:
    # Columns are strings
    column_names = dataColumns

print(f"Columns: {column_names}\n")

# Create DataFrame
df = pd.DataFrame(dataBlock, columns=column_names)

print(f"✓ Created DataFrame: {df.shape}")
print(f"\nFirst 3 rows:")
print(df.head(3))

# Add label
df['label'] = 'normal'

# Save
df.to_csv('normal_air_1.csv', index=False)
print(f"\n✓ Saved to normal_air_1.csv")

Parsing BME688 data...
Columns: ['Sensor Index', 'Sensor ID', 'Time Since PowerOn', 'Real time clock', 'Temperature', 'Pressure', 'Relative Humidity', 'Resistance Gassensor', 'Heater Profile Step Index', 'Scanning Mode Enabled', 'Scanning Cycle Index', 'Label Tag', 'Error Code']

✓ Created DataFrame: (4494, 13)

First 3 rows:
   Sensor Index  Sensor ID  Time Since PowerOn  Real time clock  Temperature  \
0             0  480007249                4551       1722344704    26.003448   
1             1  479990353                4556       1722344704    26.091400   
2             2  480026197                4561       1722344704    25.892298   

     Pressure  Relative Humidity  Resistance Gassensor  \
0  968.145264          34.951469           5684.846191   
1  968.479187          35.486801           5684.846191   
2  968.175110          34.900684           5684.846191   

   Heater Profile Step Index  Scanning Mode Enabled  Scanning Cycle Index  \
0                          0             

In [8]:
import json
import pandas as pd

print("\n" + "="*80)
print("PARSING BME688 FILES - SIMPLIFIED VERSION")
print("="*80)

def parse_simple(filepath, label):
    """Simple parser that definitely works"""
    print(f"\nParsing: {filepath}")
    
    with open(filepath, 'r') as f:
        data = json.load(f)
    
    # Get data
    dataBlock = data['rawDataBody']['dataBlock']
    dataColumns = data['rawDataBody']['dataColumns']
    
    # Extract column names from dicts
    column_names = []
    for col in dataColumns:
        if isinstance(col, dict) and 'name' in col:
            column_names.append(col['name'])
        elif isinstance(col, str):
            column_names.append(col)
        else:
            column_names.append(str(col))
    
    # Create DataFrame
    df = pd.DataFrame(dataBlock, columns=column_names)
    
    # Convert to numeric
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        except:
            pass
    
    # Add label
    df['label'] = label
    
    print(f"  ✓ Shape: {df.shape}")
    print(f"  ✓ Columns: {list(df.columns)}")
    
    return df

# Parse all three files
dfs = {}

try:
    df_normal = parse_simple('Normal_air_1.bmerawdata', 'normal')
    df_normal.to_csv('normal_air_1.csv', index=False)
    dfs['normal'] = df_normal
    print(f"  ✓ Saved: normal_air_1.csv")
except Exception as e:
    print(f"  ✗ Error: {e}")

try:
    df_acetone = parse_simple('Aceton_1.bmerawdata', 'acetone')
    df_acetone.to_csv('acetone.csv', index=False)
    dfs['acetone'] = df_acetone
    print(f"  ✓ Saved: acetone.csv")
except Exception as e:
    print(f"  ✗ Error: {e}")

try:
    df_alcohol = parse_simple('Redidlo_11_2ml.bmerawdata', 'alcohol')
    df_alcohol.to_csv('alcohol.csv', index=False)
    dfs['alcohol'] = df_alcohol
    print(f"  ✓ Saved: alcohol.csv")
except Exception as e:
    print(f"  ✗ Error: {e}")

# Combine all
if len(dfs) > 0:
    df_combined = pd.concat(list(dfs.values()), ignore_index=True)
    df_combined.to_csv('combined_data.csv', index=False)
    print(f"\n✓ Combined data saved: combined_data.csv ({len(df_combined)} rows)")

print("\n" + "="*80)
print("PARSING COMPLETE!")
print("="*80)

# Verify
for filename in ['normal_air_1.csv', 'acetone.csv', 'alcohol.csv']:
    try:
        df = pd.read_csv(filename)
        print(f"\n✓ {filename}: {df.shape[0]} rows × {df.shape[1]} columns")
    except:
        pass


PARSING BME688 FILES - SIMPLIFIED VERSION

Parsing: Normal_air_1.bmerawdata
  ✓ Shape: (4494, 14)
  ✓ Columns: ['Sensor Index', 'Sensor ID', 'Time Since PowerOn', 'Real time clock', 'Temperature', 'Pressure', 'Relative Humidity', 'Resistance Gassensor', 'Heater Profile Step Index', 'Scanning Mode Enabled', 'Scanning Cycle Index', 'Label Tag', 'Error Code', 'label']
  ✓ Saved: normal_air_1.csv

Parsing: Aceton_1.bmerawdata
  ✓ Shape: (4645, 14)
  ✓ Columns: ['Sensor Index', 'Sensor ID', 'Time Since PowerOn', 'Real time clock', 'Temperature', 'Pressure', 'Relative Humidity', 'Resistance Gassensor', 'Heater Profile Step Index', 'Scanning Mode Enabled', 'Scanning Cycle Index', 'Label Tag', 'Error Code', 'label']
  ✓ Saved: acetone.csv

Parsing: Redidlo_11_2ml.bmerawdata
  ✓ Shape: (4504, 14)
  ✓ Columns: ['Sensor Index', 'Sensor ID', 'Time Since PowerOn', 'Real time clock', 'Temperature', 'Pressure', 'Relative Humidity', 'Resistance Gassensor', 'Heater Profile Step Index', 'Scanning Mode 

In [11]:
import pandas as pd
import numpy as np

# Load all three datasets
df_normal = pd.read_csv('normal_air_1.csv')
df_acetone = pd.read_csv('acetone.csv')
df_alcohol = pd.read_csv('alcohol.csv')

# Inspect each dataset
for name, df in [('Normal', df_normal), ('Acetone', df_acetone), ('Alcohol', df_alcohol)]:
    print(f"\n=== {name} Air ===")
    print(f"Records: {len(df)}")
    print(f"Duration: {len(df) / 1.0} seconds (at 1 Hz)")
    print(f"\nTemperature - Mean: {df['temperature'].mean():.2f}°C, Std: {df['temperature'].std():.2f}°C")
    print(f"Humidity - Mean: {df['humidity'].mean():.2f}%, Std: {df['humidity'].std():.2f}%")
    print(f"Pressure - Mean: {df['pressure'].mean():.2f}hPa, Std: {df['pressure'].std():.2f}hPa")
    print(f"Gas Resistance - Mean: {df['gas_resistance'].mean():.0f}Ω, Std: {df['gas_resistance'].std():.0f}Ω")
    print(f"Gas Resistance Range: {df['gas_resistance'].min():.0f} - {df['gas_resistance'].max():.0f}Ω")
    print(f"\nMissing values:\n{df.isnull().sum()}")


=== Normal Air ===
Records: 4494
Duration: 4494.0 seconds (at 1 Hz)


KeyError: 'temperature'

In [12]:
import pandas as pd

df = pd.read_csv('combined_data.csv')

print("Data loaded successfully!")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
print(df.head())

print(f"\nTemperature range: {df['temperature'].min():.2f} - {df['temperature'].max():.2f}°C")
print(f"Humidity range: {df['humidity'].min():.2f} - {df['humidity'].max():.2f}%")
print(f"Pressure range: {df['pressure'].min():.2f} - {df['pressure'].max():.2f}hPa")
print(f"Gas Resistance range: {df['gas_resistance'].min():.0f} - {df['gas_resistance'].max():.0f}Ω")

print(f"\nClass distribution:")
print(df['label'].value_counts())

Data loaded successfully!
Shape: (13643, 14)

Columns: ['Sensor Index', 'Sensor ID', 'Time Since PowerOn', 'Real time clock', 'Temperature', 'Pressure', 'Relative Humidity', 'Resistance Gassensor', 'Heater Profile Step Index', 'Scanning Mode Enabled', 'Scanning Cycle Index', 'Label Tag', 'Error Code', 'label']

First 5 rows:
   Sensor Index  Sensor ID  Time Since PowerOn  Real time clock  Temperature  \
0             0  480007249                4551       1722344704    26.003448   
1             1  479990353                4556       1722344704    26.091400   
2             2  480026197                4561       1722344704    25.892298   
3             3  479999824                4565       1722344704    26.042248   
4             4  479999071                4569       1722344704    26.142984   

     Pressure  Relative Humidity  Resistance Gassensor  \
0  968.145264          34.951469           5684.846191   
1  968.479187          35.486801           5684.846191   
2  968.175110     

KeyError: 'temperature'

In [13]:
import pandas as pd

# ============================================================================
# RENAME COLUMNS IN CSV FILES TO STANDARD NAMES
# ============================================================================

print("\n" + "="*80)
print("RENAMING COLUMNS IN CSV FILES")
print("="*80)

# Define column mapping
column_mapping = {
    'Sensor Index': 'sensor_index',
    'Sensor ID': 'sensor_id',
    'Time Since PowerOn': 'time_since_poweron',
    'Real time clock': 'timestamp',
    'Temperature': 'temperature',
    'Pressure': 'pressure',
    'Relative Humidity': 'humidity',
    'Resistance Gassensor': 'gas_resistance',
    'Heater Profile Step Index': 'heater_profile_step',
    'Scanning Mode Enabled': 'scanning_mode',
    'Scanning Cycle Index': 'scanning_cycle',
    'Label Tag': 'label_tag',
    'Error Code': 'error_code'
}

# Process each CSV file
csv_files = ['normal_air_1.csv', 'acetone.csv', 'alcohol.csv']

for filename in csv_files:
    try:
        print(f"\nProcessing: {filename}")
        
        # Read CSV
        df = pd.read_csv(filename)
        print(f"  Original columns: {list(df.columns)}")
        
        # Rename columns
        df = df.rename(columns=column_mapping)
        print(f"  Renamed columns: {list(df.columns)}")
        
        # Save back
        df.to_csv(filename, index=False)
        print(f"  ✓ Saved: {filename} ({df.shape[0]} rows, {df.shape[1]} columns)")
        
    except FileNotFoundError:
        print(f"  ⚠ File not found: {filename}")
    except Exception as e:
        print(f"  ✗ Error: {e}")

# Combine all files
print("\n" + "="*80)
print("COMBINING ALL DATA")
print("="*80)

dfs = []
for filename in csv_files:
    try:
        df = pd.read_csv(filename)
        dfs.append(df)
        print(f"✓ Loaded {filename}: {df.shape[0]} rows")
    except:
        pass

if len(dfs) > 0:
    df_combined = pd.concat(dfs, ignore_index=True)
    df_combined.to_csv('combined_data.csv', index=False)
    print(f"\n✓ Combined dataset: {df_combined.shape[0]} rows × {df_combined.shape[1]} columns")
    print(f"✓ Saved: combined_data.csv")
    
    print(f"\nClass distribution:")
    print(df_combined['label'].value_counts())

# Verify and show sample
print("\n" + "="*80)
print("VERIFICATION - SAMPLE DATA")
print("="*80)

try:
    df = pd.read_csv('combined_data.csv')
    
    print(f"\nDataFrame shape: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    
    print(f"\nData types:")
    print(df.dtypes)
    
    print(f"\nFirst 3 rows:")
    print(df.head(3))
    
    print(f"\nBasic statistics:")
    print(df[['temperature', 'humidity', 'pressure', 'gas_resistance']].describe())
    
    print("\n" + "="*80)
    print("✓ READY FOR ANOMALY DETECTION!")
    print("="*80)
    
except Exception as e:
    print(f"\n✗ Error: {e}")


RENAMING COLUMNS IN CSV FILES

Processing: normal_air_1.csv
  Original columns: ['Sensor Index', 'Sensor ID', 'Time Since PowerOn', 'Real time clock', 'Temperature', 'Pressure', 'Relative Humidity', 'Resistance Gassensor', 'Heater Profile Step Index', 'Scanning Mode Enabled', 'Scanning Cycle Index', 'Label Tag', 'Error Code', 'label']
  Renamed columns: ['sensor_index', 'sensor_id', 'time_since_poweron', 'timestamp', 'temperature', 'pressure', 'humidity', 'gas_resistance', 'heater_profile_step', 'scanning_mode', 'scanning_cycle', 'label_tag', 'error_code', 'label']
  ✓ Saved: normal_air_1.csv (4494 rows, 14 columns)

Processing: acetone.csv
  Original columns: ['Sensor Index', 'Sensor ID', 'Time Since PowerOn', 'Real time clock', 'Temperature', 'Pressure', 'Relative Humidity', 'Resistance Gassensor', 'Heater Profile Step Index', 'Scanning Mode Enabled', 'Scanning Cycle Index', 'Label Tag', 'Error Code', 'label']
  Renamed columns: ['sensor_index', 'sensor_id', 'time_since_poweron', 't